#**Introducción a la Epidemiología y su Importancia**

La epidemiología es la disciplina que estudia la frecuencia, distribución y determinantes de los eventos relacionados con la salud en las poblaciones. Su importancia radica en que permite comprender cómo y por qué ocurren las enfermedades, orientar la toma de decisiones clínicas y de salud pública, y evaluar intervenciones.
En el contexto clínico y de investigación, la epidemiología proporciona herramientas cuantitativas esenciales para analizar datos. Entre ellas destacan:

Prevalencia de un evento, que describe la proporción de individuos afectados en un momento determinado.
Incidencia acumulada o riesgo, que indica la probabilidad de desarrollar un evento en un periodo de tiempo.
Riesgo relativo, que compara la probabilidad de un evento entre dos grupos.
Diferencia de riesgos, que muestra el exceso de riesgo atribuible a una exposición.
Odds ratio, especialmente útil en estudios de casos y controles.
Riesgo atribuible en expuestos y fracción atribuible, que permiten estimar el impacto de un factor de riesgo en una población.

Además, en estudios donde el tiempo de seguimiento varía entre individuos, se utiliza la tasa de incidencia basada en tiempo-persona, que ofrece una medida más precisa de ocurrencia.
Para organizar y analizar estos datos, se emplean herramientas como la tabla 2x2, que resume la relación entre exposición y evento, y facilita el cálculo de múltiples medidas epidemiológicas.
En diseños específicos como los estudios de casos y controles, el odds ratio es la medida principal de asociación.
Por otro lado, en el ámbito clínico, la epidemiología también se aplica en la evaluación de pruebas mediante medidas de desempeño diagnóstico, como sensibilidad, especificidad y valores predictivos.
Durante este curso, utilizaremos R como herramienta para calcular todas estas medidas, mediante funciones específicas que permitirán automatizar tanto el análisis de tablas 2x2 como la evaluación de pruebas diagnósticas.

In [ ]:
# Solo Colab
install.packages("tidyverse")
install.packages("janitor")

#---------------------------------

library(tidyverse)
library(janitor)

#---------------------------------

setwd("/content/data")


# ----------------------------------------------------------
# 1. Cargar datos de cohorte
# ----------------------------------------------------------

cohorte <- read_csv("cohorte_epidemiologia.csv") %>%
  clean_names()

head(cohorte)

In [ ]:
# Creamos una tabla 2x2 :
#                 Enfermo    No enfermo
# Expuesto           a            b
# No expuesto        c            d

tabla <- table(cohorte$exposicion, cohorte$evento)
tabla

In [ ]:
# Asegurar orden:
tabla <- table(
  factor(cohorte$exposicion, levels = c("Expuesto", "No expuesto")),
  factor(cohorte$evento, levels = c("Enfermo", "No enfermo"))
)

tabla

a <- tabla["Expuesto", "Enfermo"]
b <- tabla["Expuesto", "No enfermo"]
c <- tabla["No expuesto", "Enfermo"]
d <- tabla["No expuesto", "No enfermo"]

a; b; c; d


#**Prevalencia simple de evento**
La prevalencia es una medida epidemiológica que describe la proporción de individuos que presentan un evento o enfermedad en una población en un momento determinado. Es especialmente útil para entender la carga de enfermedad y planificar recursos en salud.
A diferencia de la incidencia, la prevalencia no considera el tiempo de aparición del evento, sino únicamente si el individuo ya presenta o no la condición al momento del análisis. Por ello, es común utilizarla en estudios transversales.
En R, la prevalencia puede calcularse de forma sencilla como la proporción de sujetos con el evento respecto al total de la población:



>prevalencia <- sum(cohorte$evento == "Enfermo") / nrow(cohorte)
prevalencia


Esta medida permite responder preguntas como: ¿qué tan frecuente es la enfermedad en esta población?, siendo un primer acercamiento clave para el análisis epidemiológico.



#**Incidencia acumulada o riesgo**

La incidencia acumulada, también conocida como riesgo, es una medida epidemiológica que estima la probabilidad de que un individuo desarrolle un evento o enfermedad durante un periodo de tiempo determinado. A diferencia de la prevalencia, esta medida sí incorpora la noción de seguimiento en el tiempo.
El riesgo suele compararse entre grupos, por ejemplo, entre expuestos y no expuestos a un factor, lo que permite empezar a evaluar posibles asociaciones causales.
En R, utilizando una tabla 2x2 clásica, el cálculo del riesgo se realiza de la siguiente manera:



> riesgo_expuestos <- a / (a + b)
> riesgo_no_expuestos <- c / (c + d)

> riesgo_expuestos
> riesgo_no_expuestos

Donde:

a: expuestos con evento
b: expuestos sin evento
c: no expuestos con evento
d: no expuestos sin evento


#**Interpretación:**

Riesgo en expuestos: probabilidad de presentar el evento en quienes tienen la exposición.
Riesgo en no expuestos: probabilidad de presentar el evento en quienes no tienen la exposición.

Esta medida permite responder preguntas como: ¿qué tan probable es que ocurra el evento en cada grupo?, siendo fundamental para el análisis epidemiológico y el cálculo de medidas de asociación en los siguientes pasos.

#**Riesgo relativo**
El riesgo relativo (RR) es una medida de asociación epidemiológica que compara el riesgo de presentar un evento entre dos grupos: expuestos y no expuestos. Permite evaluar si una exposición podría estar relacionada con un aumento o disminución del riesgo de enfermedad.
Se interpreta como cuántas veces es más (o menos) probable que ocurra el evento en el grupo expuesto en comparación con el grupo no expuesto.
En R, el cálculo es directo a partir de los riesgos previamente obtenidos:

>rr <- riesgo_expuestos / riesgo_no_expuestos
>rr

**Interpretación:**

rr = 1: no diferencia de riesgo.

rr > 1: mayor riesgo en expuestos.

rr < 1: menor riesgo en expuestos.


Esta medida es fundamental porque permite responder preguntas como: ¿la exposición aumenta el riesgo de enfermedad? y es uno de los pilares en la interpretación de estudios epidemiológicos.

#**Diferencia de riesgos**

La diferencia de riesgos es una medida de asociación que expresa el cambio absoluto en el riesgo de un evento entre dos grupos: expuestos y no expuestos. A diferencia del riesgo relativo, que es una medida proporcional, esta medida muestra el exceso (o reducción) real de riesgo atribuible a la exposición.
Es especialmente útil en la práctica clínica, ya que permite dimensionar el impacto real de una intervención o factor de riesgo en términos absolutos.
En R, se calcula de la siguiente manera:

>diferencia_riesgos <- riesgo_expuestos - riesgo_no_expuestos

>diferencia_riesgos

Si la exposición aumenta riesgo, se puede interpretar como exceso absoluto de riesgo.

Número necesario para dañar o tratar:

>nnh_nnt <- 1 / abs(diferencia_riesgos)

>nnh_nnt

**Interpretación:**

Un valor positivo indica que la exposición aumenta el riesgo.
Un valor negativo indica un efecto protector.

A partir de esta medida, se puede calcular el Número Necesario a Tratar (NNT) o Número Necesario para Dañar (NNH), que indican cuántos pacientes deben exponerse para producir (o prevenir) un evento adicional.
Esta medida responde a una pregunta clave en clínica: ¿cuántos eventos adicionales ocurren debido a la exposición?, lo que la hace muy intuitiva para la toma de decisiones.

#**Odds ratio**

El odds ratio (OR) es una medida de asociación que compara la probabilidad relativa (odds) de que ocurra un evento entre dos grupos: expuestos y no expuestos. A diferencia del riesgo relativo, el OR utiliza razones de probabilidades en lugar de riesgos directos.
Es especialmente importante en estudios de casos y controles, donde no es posible calcular riesgos directamente, pero también se usa en otros diseños.
En R, puede calcularse a partir de la tabla 2x2 como:

>or <- (a * d) / (b * c)
>or

**Interpretación**
* Un OR mayor a 1 sugiere que la exposición podría estar asociada a un mayor evento.
* Un OR menor a 1 sugiere un posible efecto protector.

Cuando el evento es poco frecuente, el odds ratio puede aproximarse al riesgo relativo, lo que facilita su interpretación.
Esta medida permite responder: ¿la exposición está asociada con el evento, especialmente cuando no podemos medir directamente el riesgo?


In [ ]:
odds_expuestos <- a / b
odds_no_expuestos <- c / d
or <- odds_expuestos / odds_no_expuestos
or

# Equivalente:
or_formula <- (a * d) / (b * c)
or_formula

#**Riesgo atribuible en expuestos y fracción atribuible**

El riesgo atribuible en expuestos es una medida que indica el exceso de riesgo que se puede atribuir directamente a una exposición en el grupo expuesto. Es decir, cuantifica cuánto del riesgo observado en los expuestos se debe realmente a la exposición.
Por otro lado, la fracción atribuible en expuestos expresa este mismo concepto en términos relativos, indicando qué proporción del evento en los expuestos es atribuible a la exposición.
En R, estas medidas se calculan de la siguiente manera:


In [ ]:
riesgo_atribuible_expuestos <- riesgo_expuestos - riesgo_no_expuestos
fraccion_atribuible_expuestos <- (rr - 1) / rr

riesgo_atribuible_expuestos
fraccion_atribuible_expuestos

**Interpretación**/

* El riesgo atribuible representa el exceso absoluto de riesgo debido a la exposición.
* La fracción atribuible indica el porcentaje del evento en los expuestos que podría evitarse si se eliminara la exposición.

Estas medidas son muy útiles en salud pública y clínica, ya que permiten responder:
¿cuánto del daño observado se debe realmente al factor de riesgo?, lo que ayuda a priorizar intervenciones y estrategias de prevención.


#**Tasa de incidencia con tiempo-persona**

La tasa de incidencia es una medida que evalúa la velocidad con la que ocurren nuevos eventos en una población, considerando no solo cuántos eventos ocurren, sino también el tiempo de seguimiento de cada individuo.
Esto es especialmente útil cuando los sujetos no son seguidos durante el mismo periodo, lo que es muy común en estudios reales.
En lugar de usar personas como denominador, se utiliza el concepto de tiempo-persona (por ejemplo, años-persona), que representa la suma del tiempo que cada individuo estuvo en riesgo.
En R, el cálculo puede realizarse agrupando por exposición:

In [ ]:

tasas <- cohorte %>%
  group_by(exposicion) %>%
  summarise(
    eventos = sum(evento == "Enfermo"),
    tiempo_persona = sum(tiempo_persona_anios),
    tasa_incidente = eventos / tiempo_persona
  )


tasas


razon_tasas <- tasas$tasa_incidente[tasas$exposicion == "Expuesto"] /
  tasas$tasa_incidente[tasas$exposicion == "No expuesto"]

razon_tasas



**Interpretación**

* La tasa de incidencia indica cuántos eventos ocurren por unidad de tiempo-persona.

* La razón de tasas compara la velocidad del evento entre expuestos y no expuestos.

Esta medida responde a una pregunta clave:
¿con qué rapidez ocurre el evento en cada grupo?, lo cual es fundamental cuando el tiempo de seguimiento varía entre individuos.

#**Función general para tabla 2x2**

La tabla 2x2 es una herramienta fundamental en epidemiología que permite organizar los datos según exposición (sí/no) y evento (sí/no). A partir de esta estructura, es posible calcular múltiples medidas de asociación de forma sistemática.
En la práctica, para evitar cálculos repetitivos y facilitar el análisis, es muy útil crear una función en R que integre todas las medidas vistas hasta ahora.

In [ ]:
medidas_2x2 <- function(a, b, c, d) {
  riesgo_exp <- a / (a + b)
  riesgo_no_exp <- c / (c + d)
  rr <- riesgo_exp / riesgo_no_exp
  dr <- riesgo_exp - riesgo_no_exp
  or <- (a * d) / (b * c)

  tibble(
    a_enfermos_expuestos = a,
    b_no_enfermos_expuestos = b,
    c_enfermos_no_expuestos = c,
    d_no_enfermos_no_expuestos = d,
    riesgo_expuestos = riesgo_exp,
    riesgo_no_expuestos = riesgo_no_exp,
    riesgo_relativo = rr,
    odds_ratio = or,
    diferencia_riesgos = dr,
    nnt_o_nnh = 1 / abs(dr),
    fraccion_atribuible_expuestos = (rr - 1) / rr
  )
}

medidas_2x2(a, b, c, d)


**Interpretación**

Esta función permite obtener en un solo paso:

Riesgos en ambos grupos
Riesgo relativo
Diferencia de riesgos
Odds ratio
Número necesario a tratar o dañar
Fracción atribuible

Su utilidad principal es automatizar el análisis epidemiológico, reducir errores y facilitar la interpretación en estudios clínicos y de investigación.
Responde de manera integrada a la pregunta:
¿qué relación existe entre la exposición y el evento desde diferentes perspectivas?

#**Casos y controles: odds ratio**

Los estudios de casos y controles son un diseño epidemiológico en el que se comparan individuos con una enfermedad (casos) y sin ella (controles), evaluando retrospectivamente su exposición a un posible factor de riesgo.
En este tipo de estudios no es posible calcular directamente el riesgo o la incidencia, por lo que la medida principal de asociación es el odds ratio (OR).
En R, primero organizamos los datos en una tabla 2x2, donde se cruzan la exposición y la condición de caso/control:

In [ ]:
casos_controles <- read_csv("data/casos_controles_demo.csv") %>%
  clean_names()

tabla_cc <- table(
  factor(casos_controles$exposicion, levels = c("Expuesto", "No expuesto")),
  factor(casos_controles$enfermedad, levels = c("Caso", "Control"))
)

tabla_cc

# En casos y controles:
#                 Caso    Control
# Expuesto          a        b
# No expuesto       c        d

a_cc <- tabla_cc["Expuesto", "Caso"]
b_cc <- tabla_cc["Expuesto", "Control"]
c_cc <- tabla_cc["No expuesto", "Caso"]
d_cc <- tabla_cc["No expuesto", "Control"]

or_cc <- (a_cc * d_cc) / (b_cc * c_cc)
or_cc

# Nota
# En estudios de casos y controles se estima OR, no riesgo directo,
# porque el número de casos y controles fue fijado por diseño.

**Interpretación**

La tabla resultante muestra:

Filas: estado de exposición (Expuesto / No expuesto)
Columnas: estado de enfermedad (Caso / Control)

A partir de esta tabla se calcula el odds ratio, que indica la asociación entre exposición y enfermedad.

OR = 1: no hay asociación
OR > 1: la exposición se asocia con mayor probabilidad de enfermedad
OR < 1: la exposición puede ser protectora

Este enfoque permite responder:
¿los casos estuvieron más expuestos que los controles?, lo cual es clave cuando se estudian enfermedades raras o con largos periodos de aparición.

#**Medidas de prueba diagnóstica**

Las medidas de prueba diagnóstica se utilizan para evaluar qué tan bien una prueba identifica correctamente a los pacientes enfermos y sanos. Son fundamentales en la práctica clínica para interpretar resultados y tomar decisiones médicas.
Estas medidas se basan en una tabla 2x2 donde se comparan los resultados de la prueba con el estado real de enfermedad:

VP (verdaderos positivos): enfermos correctamente identificados
FP (falsos positivos): sanos clasificados como enfermos
FN (falsos negativos): enfermos no detectados
VN (verdaderos negativos): sanos correctamente identificados

En R, las principales medidas se calculan así:

In [ ]:
# Ejemplo:
#                 Enfermo    Sano
# Prueba +           VP       FP
# Prueba -           FN       VN

vp <- 80
fp <- 30
fn <- 20
vn <- 170

sensibilidad <- vp / (vp + fn)
especificidad <- vn / (vn + fp)
vpp <- vp / (vp + fp)
vpn <- vn / (vn + fn)
rv_positiva <- sensibilidad / (1 - especificidad)
rv_negativa <- (1 - sensibilidad) / especificidad

tibble(
  sensibilidad = sensibilidad,
  especificidad = especificidad,
  valor_predictivo_positivo = vpp,
  valor_predictivo_negativo = vpn,
  razon_verosimilitud_positiva = rv_positiva,
  razon_verosimilitud_negativa = rv_negativa
)

**Interpretación**

* Sensibilidad: capacidad de la prueba para detectar enfermos
* Especificidad: capacidad para identificar sanos
* Valor predictivo positivo (VPP): probabilidad de estar enfermo si la prueba es positiva
* Valor predictivo negativo (VPN): probabilidad de estar sano si la prueba es negativa
* Razones de verosimilitud (RV): indican cuánto cambia la probabilidad de enfermedad tras el resultado

Estas medidas permiten responder:
¿qué tan confiable es una prueba diagnóstica en la práctica clínica?, siendo clave para interpretar estudios y tomar decisiones médicas informadas.

#**Función para prueba diagnóstica**

Para facilitar el análisis de pruebas diagnósticas y evitar repetir cálculos, es útil crear una función en R que integre todas las medidas principales en un solo paso.
Esta función permite introducir los valores de la tabla 2x2 (VP, FP, FN, VN) y obtener automáticamente los indicadores más importantes para la interpretación clínica.

In [ ]:
medidas_diagnosticas <- function(vp, fp, fn, vn) {
  sensibilidad <- vp / (vp + fn)
  especificidad <- vn / (vn + fp)
  vpp <- vp / (vp + fp)
  vpn <- vn / (vn + fn)
  rv_pos <- sensibilidad / (1 - especificidad)
  rv_neg <- (1 - sensibilidad) / especificidad

  tibble(
    sensibilidad = sensibilidad,
    especificidad = especificidad,
    vpp = vpp,
    vpn = vpn,
    rv_positiva = rv_pos,
    rv_negativa = rv_neg
  )
}

medidas_diagnosticas(vp = 80, fp = 30, fn = 20, vn = 170)

**Interpretación**

Esta función resume en una tabla:

* La capacidad de la prueba para detectar enfermos (sensibilidad)
* Su capacidad para descartar sanos (especificidad)
* La probabilidad clínica asociada a resultados positivos y negativos (VPP y VPN)
* El cambio en la probabilidad de enfermedad tras el resultado (razones de verosimilitud)

Su principal utilidad es automatizar el análisis, reducir errores y facilitar la interpretación en la práctica clínica y en investigación.
Responde a una pregunta central:
¿qué tan útil es esta prueba para apoyar decisiones clínicas?

#**Ejercicios**


 **Ejercicio 1:**
 Con a=40, b=160, c=20, d=180:
 Calcular riesgo en expuestos, riesgo en no expuestos, RR, OR y diferencia de riesgos.

 **Ejercicio 2:**
 Interpretar clínicamente un RR de 2.0 y una diferencia de riesgos de 0.10.

 **Ejercicio 3:**
 Calcular sensibilidad, especificidad, VPP y VPN para:
 VP=45, FP=15, FN=10, VN=130.